# Does data ordering matter when fine-tuning a small LLM?

**Hypothesis:** curriculum-ordered fine-tuning (easy → hard) beats shuffled fine-tuning for small models at fixed compute.

**The experiment:** fine-tune Qwen2.5-0.5B-Instruct three times on the *same* 2,000 GSM8K math problems with *identical* hyperparameters. Only the order changes:
- **Run A** — shuffled (control)
- **Run B** — easy → hard (curriculum)
- **Run C** — hard → easy (anti-curriculum — if this also beats A, the effect isn't curriculum)

Then evaluate all three on the same 200 held-out test problems and compare accuracy.

**Before running:** in Colab go to `Runtime → Change runtime type → T4 GPU`. Then `Runtime → Run all`. Total time: roughly 1.5–2 hours. If the session disconnects, adapters are saved after each run, so you can skip completed runs and resume.

In [12]:
# Cell 1: Install dependencies (~2 min)
!pip install -q -U transformers datasets peft accelerate
!pip uninstall -y torchao
import torch
assert torch.cuda.is_available(), "No GPU found! Go to Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [13]:
# Cell 2: Load GSM8K and build the three orderings
import random
from datasets import load_dataset

SEED = 42
N_TRAIN = 2000
N_EVAL = 200

gsm = load_dataset("openai/gsm8k", "main")

# Difficulty proxy: number of calculation steps (<<...>> annotations) in the reference solution.
# Crude but objective and correlates with reasoning depth.
def difficulty(example):
    return example["answer"].count("<<")

train_pool = list(gsm["train"])
random.Random(SEED).shuffle(train_pool)          # fix WHICH 2000 examples we use
train_pool = train_pool[:N_TRAIN]

eval_pool = list(gsm["test"])[:N_EVAL]           # held-out test problems, never trained on

# The three orderings of the SAME 2000 examples
order_A = list(train_pool)
random.Random(SEED + 1).shuffle(order_A)                          # A: shuffled
order_B = sorted(train_pool, key=difficulty)                      # B: easy -> hard
order_C = sorted(train_pool, key=difficulty, reverse=True)        # C: hard -> easy

diffs = sorted(difficulty(e) for e in train_pool)
print(f"Train examples: {len(train_pool)}, eval examples: {len(eval_pool)}")
print(f"Difficulty (calc steps) — min: {diffs[0]}, median: {diffs[len(diffs)//2]}, max: {diffs[-1]}")
print("First problem in B (should be short/easy):\n", order_B[0]["question"][:200])

Train examples: 2000, eval examples: 200
Difficulty (calc steps) — min: 0, median: 3, max: 8
First problem in B (should be short/easy):
 CJ, KJ, and AJ collect stamps.  CJ has 5 more than twice the number of stamps that KJ has, and KJ has half as many as AJ.  If the three boys have 930 stamps all together, how many stamps does AJ have?


In [14]:
# Cell 3: Tokenization — format as chat, train on full sequence
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_LEN = 640

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(ex):
    # Strip the <<...>> calculator annotations from the target text
    import re
    answer = re.sub(r"<<[^>]*>>", "", ex["answer"])
    messages = [
        {"role": "user", "content": "Solve this math problem step by step. End with the final answer after '####'.\n\n" + ex["question"]},
        {"role": "assistant", "content": answer},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

def tokenize_ordered(examples):
    """Tokenize a list of examples, PRESERVING ORDER."""
    texts = [format_example(e) for e in examples]
    out = tokenizer(texts, truncation=True, max_length=MAX_LEN)
    return [{"input_ids": ids, "attention_mask": mask}
            for ids, mask in zip(out["input_ids"], out["attention_mask"])]

dataset_A = tokenize_ordered(order_A)
dataset_B = tokenize_ordered(order_B)
dataset_C = tokenize_ordered(order_C)
print("Tokenized. Example length (tokens):", len(dataset_A[0]["input_ids"]))

Tokenized. Example length (tokens): 159


In [17]:
import gc
from torch.utils.data import Dataset, SequentialSampler
from transformers import (AutoModelForCausalLM, Trainer, TrainingArguments,
                          DataCollatorForLanguageModeling)
from peft import LoraConfig, get_peft_model

class ListDataset(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

class OrderedTrainer(Trainer):
    """CRITICAL: HF Trainer shuffles by default. This forces sequential order."""
    def _get_train_sampler(self, *args, **kwargs):
        return SequentialSampler(self.train_dataset)

def train_one_run(run_name, dataset):
    print(f"\n{'='*50}\nTraining run {run_name}\n{'='*50}")
    torch.manual_seed(SEED)  # same init for LoRA weights in every run

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="cuda")
    model.config.use_cache = False

    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
                      task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
    model = get_peft_model(model, lora)

    args = TrainingArguments(
        output_dir=f"./out_{run_name}",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        lr_scheduler_type="constant",   # constant LR so late examples aren't unfairly down-weighted
        warmup_steps=10,
        logging_steps=50,
        save_strategy="no",
        fp16=True,
        report_to="none",
        # group_by_length=False,          # must stay False or order is destroyed - REMOVED DUE TO TYPERROR
        seed=SEED,
    )

    trainer = OrderedTrainer(
        model=model, args=args,
        train_dataset=ListDataset(dataset),
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    trainer.train()

    model.save_pretrained(f"./adapter_{run_name}")
    del model, trainer
    gc.collect(); torch.cuda.empty_cache()
    print(f"Run {run_name} done — adapter saved to ./adapter_{run_name}")

In [20]:
# Sanity check v2: is order actually preserved?
import torch.nn as nn

class DummyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 2)
    def forward(self, **kwargs):
        return None

test_trainer = OrderedTrainer(
    model=DummyModel(),
    args=TrainingArguments(output_dir="./tmp", report_to="none"),
    train_dataset=ListDataset(list(range(100))),
)
sampler = test_trainer._get_train_sampler()
order = list(sampler)
print("First 10 indices:", order[:10])
assert order == list(range(100)), "ORDER IS BROKEN — sampler is shuffling!"
print("✓ Order preserved — experiment is valid")

First 10 indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
✓ Order preserved — experiment is valid


In [21]:
# Cell 5: Run all three trainings (~15–25 min each on T4)
import os
for name, ds in [("A_shuffled", dataset_A), ("B_easy_to_hard", dataset_B), ("C_hard_to_easy", dataset_C)]:
    if os.path.exists(f"./adapter_{name}"):
        print(f"Skipping {name} — adapter already exists (resume-friendly)")
    else:
        train_one_run(name, ds)


Training run A_shuffled


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Step,Training Loss
50,0.561150
100,0.346130
150,0.351765
200,0.362979
250,0.346949


Run A_shuffled done — adapter saved to ./adapter_A_shuffled

Training run B_easy_to_hard


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Step,Training Loss
50,0.577742
100,0.356972
150,0.339826
200,0.341713
250,0.352988


Run B_easy_to_hard done — adapter saved to ./adapter_B_easy_to_hard

Training run C_hard_to_easy


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Step,Training Loss
50,0.518391
100,0.352592
150,0.337262
200,0.331176
250,0.360014


Run C_hard_to_easy done — adapter saved to ./adapter_C_hard_to_easy


In [22]:
# Cell 6: Evaluation — same 200 held-out problems for every model (~10–15 min per model)
import re
from peft import PeftModel
from transformers import AutoModelForCausalLM

def extract_gold(answer_text):
    return answer_text.split("####")[-1].strip().replace(",", "")

def extract_pred(generated_text):
    if "####" in generated_text:
        tail = generated_text.split("####")[-1]
        nums = re.findall(r"-?\d+\.?\d*", tail.replace(",", ""))
        if nums: return nums[0].rstrip(".")
    nums = re.findall(r"-?\d+\.?\d*", generated_text.replace(",", ""))
    return nums[-1].rstrip(".") if nums else None

def norm(x):
    try: return float(x)
    except (TypeError, ValueError): return None

@torch.no_grad()
def evaluate(adapter_path, label, batch_size=8):
    print(f"\nEvaluating {label} ...")
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="cuda")
    model = PeftModel.from_pretrained(base, adapter_path) if adapter_path else base
    model.eval()

    tokenizer.padding_side = "left"
    correct = 0
    for i in range(0, len(eval_pool), batch_size):
        batch = eval_pool[i:i+batch_size]
        prompts = [tokenizer.apply_chat_template(
            [{"role": "user", "content": "Solve this math problem step by step. End with the final answer after '####'.\n\n" + ex["question"]}],
            tokenize=False, add_generation_prompt=True) for ex in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True,
                           truncation=True, max_length=512).to("cuda")
        out = model.generate(**inputs, max_new_tokens=320, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        for j, ex in enumerate(batch):
            gen = tokenizer.decode(out[j][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            if norm(extract_pred(gen)) is not None and norm(extract_pred(gen)) == norm(extract_gold(ex["answer"])):
                correct += 1
        if (i // batch_size) % 5 == 0:
            print(f"  {i+len(batch)}/{len(eval_pool)} — running accuracy: {correct/(i+len(batch)):.1%}")

    del model, base
    gc.collect(); torch.cuda.empty_cache()
    acc = correct / len(eval_pool)
    print(f"{label}: {correct}/{len(eval_pool)} = {acc:.1%}")
    return acc

results = {}
results["baseline (no fine-tune)"] = evaluate(None, "baseline (no fine-tune)")
results["A: shuffled"]            = evaluate("./adapter_A_shuffled", "A: shuffled")
results["B: easy \u2192 hard"]      = evaluate("./adapter_B_easy_to_hard", "B: easy → hard")
results["C: hard \u2192 easy"]      = evaluate("./adapter_C_hard_to_easy", "C: hard → easy")


Evaluating baseline (no fine-tune) ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  8/200 — running accuracy: 37.5%
  48/200 — running accuracy: 31.2%
  88/200 — running accuracy: 35.2%
  128/200 — running accuracy: 39.1%
  168/200 — running accuracy: 41.1%
baseline (no fine-tune): 86/200 = 43.0%

Evaluating A: shuffled ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  8/200 — running accuracy: 37.5%
  48/200 — running accuracy: 33.3%
  88/200 — running accuracy: 38.6%
  128/200 — running accuracy: 37.5%
  168/200 — running accuracy: 35.7%
A: shuffled: 68/200 = 34.0%

Evaluating B: easy → hard ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  8/200 — running accuracy: 37.5%
  48/200 — running accuracy: 25.0%
  88/200 — running accuracy: 31.8%
  128/200 — running accuracy: 34.4%
  168/200 — running accuracy: 32.7%
B: easy → hard: 61/200 = 30.5%

Evaluating C: hard → easy ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  8/200 — running accuracy: 25.0%
  48/200 — running accuracy: 20.8%
  88/200 — running accuracy: 33.0%
  128/200 — running accuracy: 31.2%
  168/200 — running accuracy: 32.7%
C: hard → easy: 64/200 = 32.0%


In [23]:
# Cell 7: Final results
print("\n" + "="*44)
print("FINAL RESULTS — GSM8K accuracy, 200 problems")
print("="*44)
for k, v in results.items():
    print(f"{k:<28} {v:.1%}")
print("="*44)
print("\nHow to read this:")
print("- B > A by 5%+ (10+ problems) AND C <= A  -> evidence FOR curriculum ordering")
print("- B ≈ A ≈ C (within ~2%)                  -> ordering doesn't matter at this scale")
print("- B and C both beat A                     -> something else is going on (interesting!)")
print("\nDifferences under ~2% (4 problems) are likely noise. For a believable result,")
print("change SEED in Cell 2 (e.g. to 43) and rerun — does the pattern hold?")


FINAL RESULTS — GSM8K accuracy, 200 problems
baseline (no fine-tune)      43.0%
A: shuffled                  34.0%
B: easy → hard               30.5%
C: hard → easy               32.0%

How to read this:
- B > A by 5%+ (10+ problems) AND C <= A  -> evidence FOR curriculum ordering
- B ≈ A ≈ C (within ~2%)                  -> ordering doesn't matter at this scale
- B and C both beat A                     -> something else is going on (interesting!)

Differences under ~2% (4 problems) are likely noise. For a believable result,
change SEED in Cell 2 (e.g. to 43) and rerun — does the pattern hold?
